# oxDNA: LAMMPS Propeller Twist Optimization

This notebook optimizes oxDNA1 force-field parameters to match a target propeller twist using LAMMPS as the simulation engine and DiffTRe for gradient computation.

## What is mythos?

**[Mythos](https://github.com/mythos-bio/mythos)** is a differentiable molecular simulation framework for parameterizing coarse-grained force fields. It provides automatic differentiation through simulation workflows — enabling gradient-based optimization of force field parameters to match experimental or all-atom reference data.

This notebook shows how to use LAMMPS as an alternative simulation backend for oxDNA model optimization. LAMMPS provides a fast, well-tested MD engine while mythos handles the differentiable optimization on top.

## Imports

In [ ]:
import logging
import typing
from pathlib import Path

import jax
import jax.numpy as jnp
import jax_md
import optax

import mythos.energy.dna1 as dna1_energy
import mythos.observables as jd_obs
import mythos.optimization.objective as jdna_objective
import mythos.optimization.optimization as jdna_optimization
import mythos.optimization.simulator as jdna_simulator
import mythos.utils.types as jdna_types
from mythos.input import topology
from mythos.simulators.lammps.lammps_oxdna import LAMMPSoxDNASimulator
from mythos.ui.loggers.console import ConsoleLogger

jax.config.update("jax_enable_x64", True)
logging.basicConfig(level=logging.INFO)

## Configuration

In [ ]:
N_OPT_STEPS = 25
LEARNING_RATE = 1e-3
INPUT_DIR = Path("../../data/templates/simple-helix-60bp-oxdna1-lammps").resolve()
TARGET = jd_obs.propeller.TARGETS["oxDNA"]
KT = 0.1  # Must match the value used in LAMMPS

## Build the energy function and simulator

In [ ]:
top = topology.from_oxdna_file(INPUT_DIR / "sys.top")

energy_fn = dna1_energy.create_default_energy_fn(
    topology=top,
).with_noopt(
    "ss_stack_weights", "ss_hb_weights"
).without_terms(
    "BondedExcludedVolume"
)

simulator = LAMMPSoxDNASimulator(
    input_dir=INPUT_DIR,
    energy_fn=energy_fn,
)


def simulator_fn(params, _meta):
    return [simulator.run(params)]


obs_trajectory = "trajectory"

trajectory_simulator = jdna_simulator.BaseSimulator(
    name="oxdna-sim",
    fn=simulator_fn,
    exposes=[obs_trajectory],
    meta_data={},
)

## Define the propeller twist objective

In [ ]:
prop_twist_fn = jd_obs.propeller.PropellerTwist(
    rigid_body_transform_fn=energy_fn.energy_fns[0].transform_fn,
    h_bonded_base_pairs=jnp.array([[1, 14], [2, 13], [3, 12], [4, 11], [5, 10], [6, 9]]),
)


def prop_twist_loss_fn(traj, weights, *_args, **_kwargs):
    obs = prop_twist_fn(traj)
    expected_prop_twist = jnp.dot(weights, obs)
    loss = jnp.sqrt((expected_prop_twist - TARGET) ** 2)
    return loss, (("prop_twist", expected_prop_twist), {})


opt_params = energy_fn.opt_params()

propeller_twist_objective = jdna_objective.DiffTReObjective(
    name="DiffTRe",
    required_observables=[obs_trajectory],
    needed_observables=[obs_trajectory],
    logging_observables=["loss", "prop_twist"],
    grad_or_loss_fn=prop_twist_loss_fn,
    energy_fn=energy_fn,
    min_n_eff_factor=0.95,
    n_equilibration_steps=1000,
    max_valid_opt_steps=100,
)

## Run the optimization

Using `SimpleOptimizer.run()` to manage the optimization loop.

In [ ]:
opt = jdna_optimization.SimpleOptimizer(
    objective=propeller_twist_objective,
    simulator=trajectory_simulator,
    optimizer=optax.adam(learning_rate=LEARNING_RATE),
    logger=ConsoleLogger(),
)

output = opt.run(params=opt_params, n_steps=N_OPT_STEPS)
print("\nOptimization complete!")
print(f"Final params: {output.opt_params}")